# Snowflake Security Configuration Validation Notebook

This notebook provides a guide to Snowflake security configuration with executable validation commands. It is designed to run in Snowflake Notebooks (Workspaces) and validates security settings across all Snowflake editions. It is intended to help Snowflake administrators make informed choices about the security features included with their deployment.

> **Important:** This resource is not a substitute for a thorough, services-led deployment, nor does it supersede any other obligations you may have to Snowflake, your organization, your outside regulators, or other bodies to which you owe compliance or conformance. The checklists and code samples contained in this document are informational only.

**Conventions**
- Every check is read-only (`SHOW`, `DESCRIBE`, `SELECT`, `SYSTEM$` information functions). Remediation SQL is shown commented out or in markdown so nothing changes unless you run it deliberately.
- `SNOWFLAKE.ACCOUNT_USAGE` views can lag by up to 2 hours (`LOGIN_HISTORY` up to 2 hours, `POLICY_REFERENCES` up to 2 hours).
- Status columns use ✅ (matches recommendation), ⚠️ (review), and ❌ (likely gap).

## Prerequisites and Setup

The checks read account-level parameters, `SNOWFLAKE.ACCOUNT_USAGE`, and security integrations, so run the notebook as **ACCOUNTADMIN**, or as a custom role that has been granted:
- `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (or the `SNOWFLAKE.SECURITY_VIEWER` / `SNOWFLAKE.GOVERNANCE_VIEWER` database roles)
- `MONITOR` on the account and `USAGE` on the integrations you want to inspect

A few functions (`SYSTEM$GET_CMK_INFO`, `SYSTEM$INTERNAL_STAGES_PUBLIC_ACCESS_STATUS`, `SYSTEM$GET_PRIVATELINK_AUTHORIZED_ENDPOINTS`) require ACCOUNTADMIN.

In [ ]:
%%sql -r ROLE_CONTEXT
USE ROLE ACCOUNTADMIN;
SELECT CURRENT_ROLE() AS current_role, CURRENT_USER() AS current_user;

## Section 0: Start with Trust Center

[Trust Center](https://docs.snowflake.com/en/user-guide/trust-center/overview) continuously evaluates your account against the **Security Essentials** and **CIS Snowflake Benchmark** scanner packages (and optionally Threat Intelligence). Many items in this notebook are also covered by Trust Center scanners; use this notebook to understand *why* each control matters and to dig into the underlying configuration, and use Trust Center for ongoing monitoring.

Enable scanner packages in Snowsight under **Monitoring » Trust Center » Scanner Packages**.

In [ ]:
%%sql -r TRUST_CENTER_OPEN_FINDINGS
-- Open Trust Center findings by severity (empty if no scanner packages are enabled)
SELECT
    severity,
    scanner_package_id,
    scanner_id,
    COUNT(*)                  AS open_findings,
    SUM(at_risk_entity_count) AS at_risk_entities,
    MAX(updated_on)           AS last_updated
FROM SNOWFLAKE.ACCOUNT_USAGE.TRUST_CENTER_FINDINGS
WHERE state IS NULL OR UPPER(state) = 'OPEN'
GROUP BY ALL
ORDER BY CASE severity WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END,
         open_findings DESC;

## Section 1: General Security Considerations (All Editions)

### 1.1 Organization and Account Name Validation

Organization and account names are visible in DNS (`<org>-<account>.snowflakecomputing.com`). Choose obfuscated names if organizational policy requires hiding publicly visible assets.

In [ ]:
%%sql -r ACCOUNT_INFO
SELECT
    CURRENT_ACCOUNT()           AS account_locator,
    CURRENT_ACCOUNT_NAME()      AS account_name,
    CURRENT_ORGANIZATION_NAME() AS organization_name,
    CURRENT_REGION()            AS region,
    CURRENT_VERSION()           AS snowflake_version;

### 1.2 Network Policy and Network Rule Validation

Apply an account-level network policy as a baseline, then layer more granular policies on specific users (especially service users), security integrations, and admin users. Build policies from **network rules** rather than the legacy `ALLOWED_IP_LIST` / `BLOCKED_IP_LIST` properties, and use [Snowflake-managed network rules](https://docs.snowflake.com/en/user-guide/network-rules) (e.g. for SaaS integrations) where available.

Precedence: user-level policy > security-integration policy > account-level policy.

In [ ]:
%%sql -r NETWORK_POLICY_DISCOVERY
SHOW NETWORK POLICIES
->> SELECT
        "name"                             AS policy_name,
        "entries_in_allowed_network_rules" AS allowed_rules,
        "entries_in_blocked_network_rules" AS blocked_rules,
        "entries_in_allowed_ip_list"       AS legacy_allowed_ips,
        "entries_in_blocked_ip_list"       AS legacy_blocked_ips,
        CASE WHEN "entries_in_allowed_ip_list" > 0 OR "entries_in_blocked_ip_list" > 0
             THEN '⚠️ Uses legacy IP lists - migrate to network rules'
             ELSE '✅ Network-rule based' END AS status,
        "comment"
    FROM $1;

In [ ]:
%%sql -r NETWORK_POLICY_ASSIGNMENTS
-- Where network policies are attached (account, users, integrations)
SELECT
    policy_name,
    ref_entity_domain,
    ref_entity_name,
    policy_status
FROM SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES
WHERE policy_kind = 'NETWORK_POLICY'
ORDER BY ref_entity_domain, ref_entity_name;

In [ ]:
%%sql -r ACCOUNT_LVL_NETWORK_POLICY
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN ACCOUNT
->> SELECT "key", "value",
        IFF("value" = '' OR "value" IS NULL, '❌ No account-level network policy', '✅ Account policy set') AS status
    FROM $1;

In [ ]:
%%sql -r NETWORK_RULE_VALIDATION
SHOW NETWORK RULES IN ACCOUNT;

In [ ]:
%%sql -r SERVICE_USERS_WITHOUT_NETPOLICY
-- Service users with no user-level network policy (they still inherit any account policy)
SELECT u.name, u.type, u.has_rsa_public_key, u.has_pat, u.has_workload_identity, u.last_success_login
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS u
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES p
       ON p.policy_kind = 'NETWORK_POLICY'
      AND p.ref_entity_domain = 'USER'
      AND p.ref_entity_name = u.name
WHERE u.deleted_on IS NULL
  AND u.disabled = FALSE
  AND u.type IN ('SERVICE', 'LEGACY_SERVICE')
  AND p.policy_name IS NULL
ORDER BY u.name;

### 1.3 User Provisioning and User Hygiene

Provision users from your identity provider with SCIM where possible. SCIM-provisioned users are typically owned by the SCIM provisioner role (e.g. `AAD_PROVISIONER`, `OKTA_PROVISIONER`, `GENERIC_SCIM_PROVISIONER`), and `PROVISIONED_BY` indicates how a user was created.

Set the `TYPE` property on every user: `PERSON` for humans, `SERVICE` for workloads. `LEGACY_SERVICE` is being retired (see 1.7).

In [ ]:
%%sql -r SCIM_INTEGRATIONS
SHOW SECURITY INTEGRATIONS
->> SELECT "name", "type", "enabled", "created_on" FROM $1 WHERE "type" ILIKE '%SCIM%';

In [ ]:
%%sql -r USER_PROVISIONING_SUMMARY
SELECT
    COALESCE(type, 'NULL (treated as PERSON)') AS user_type,
    owner,
    provisioned_by,
    COUNT(*)                                        AS users,
    COUNT_IF(disabled::BOOLEAN)                     AS disabled_users,
    COUNT_IF(last_success_login IS NULL)            AS never_logged_in,
    COUNT_IF(last_success_login < DATEADD(DAY, -90, CURRENT_TIMESTAMP())) AS inactive_90d
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL
GROUP BY ALL
ORDER BY users DESC;

In [ ]:
%%sql -r STALE_USERS
-- Enabled users with no successful login in 90 days (candidates for disabling)
SELECT name, type, owner, created_on, last_success_login, has_password, has_rsa_public_key, has_pat
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL
  AND disabled = FALSE
  AND COALESCE(last_success_login, created_on) < DATEADD(DAY, -90, CURRENT_TIMESTAMP())
ORDER BY last_success_login NULLS FIRST;

### 1.4 Session Policy Validation

A session policy defines the idle timeout (in minutes) for Snowsight (`SESSION_UI_IDLE_TIMEOUT_MINS`) and for clients (`SESSION_IDLE_TIMEOUT_MINS`). The minimum is 5 minutes; without a policy the default is 240 minutes (4 hours). Session policies can also restrict secondary roles (`ALLOWED_SECONDARY_ROLES`). Session policies are available in all editions.

Consider stricter policies for privileged users.

In [ ]:
%%sql -r SESSION_POLICIES
SHOW SESSION POLICIES IN ACCOUNT;

In [ ]:
%%sql -r SESSION_POLICY_ASSIGNMENTS
SELECT policy_name, ref_entity_domain, ref_entity_name, policy_status
FROM SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES
WHERE policy_kind = 'SESSION_POLICY'
ORDER BY ref_entity_domain, ref_entity_name;

In [ ]:
%%sql -r SESSION_PARAMETERS
SHOW PARAMETERS LIKE '%SESSION%' IN ACCOUNT;

### 1.5 Administrative Roles Audit

Grant the built-in administrative roles (ACCOUNTADMIN, SECURITYADMIN, USERADMIN, SYSADMIN, ORGADMIN) carefully to keep a least-privilege posture.
- They should never be any user's **default role**.
- Every ACCOUNTADMIN user should use strong authentication (SSO + MFA, or password + MFA). Keep at least two, and give at least one break-glass user one-time passcodes stored in a vault.
- Periodically review who holds these roles and why.

In [ ]:
%%sql -r ADMIN_ROLE_HOLDERS
-- Users holding admin roles directly, with their authentication posture
SELECT
    g.role,
    g.grantee_name               AS user_name,
    u.type,
    u.default_role,
    u.has_password,
    u.has_mfa,
    u.has_rsa_public_key,
    u.ext_authn_uid IS NOT NULL  AS has_sso_identity,
    u.last_success_login,
    CASE
        WHEN u.default_role IN ('ACCOUNTADMIN','SECURITYADMIN','ORGADMIN') THEN '❌ Admin role is default role'
        WHEN u.has_password AND NOT u.has_mfa                           THEN '❌ Password without MFA'
        ELSE '✅ OK' END            AS status
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS g
JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON u.name = g.grantee_name AND u.deleted_on IS NULL
WHERE g.deleted_on IS NULL
  AND g.role IN ('ACCOUNTADMIN','SECURITYADMIN','USERADMIN','SYSADMIN','ORGADMIN')
ORDER BY g.role, status, user_name;

In [ ]:
%%sql -r ADMIN_ROLE_INHERITANCE
-- Custom roles that have been granted an admin role (inherit its privileges)
SELECT name AS admin_role, grantee_name AS granted_to_role, granted_by, created_on
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE deleted_on IS NULL
  AND granted_on = 'ROLE'
  AND privilege = 'USAGE'
  AND name IN ('ACCOUNTADMIN','SECURITYADMIN','USERADMIN','SYSADMIN','ORGADMIN')
  AND grantee_name NOT IN ('ACCOUNTADMIN','SECURITYADMIN','USERADMIN','SYSADMIN','ORGADMIN')
ORDER BY admin_role;

In [ ]:
%%sql -r ACCOUNTADMIN_COUNT
SELECT
    COUNT(*) AS accountadmin_users,
    CASE WHEN COUNT(*) < 2  THEN '⚠️ Fewer than 2 ACCOUNTADMINs - no redundancy'
         WHEN COUNT(*) > 5  THEN '⚠️ More than 5 ACCOUNTADMINs - review need'
         ELSE '✅ OK' END AS status
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS
WHERE role = 'ACCOUNTADMIN' AND deleted_on IS NULL;

### 1.6 Data Exfiltration Controls (Stages and Unload)

| Parameter | Recommended | Purpose |
|---|---|---|
| `PREVENT_UNLOAD_TO_INLINE_URL` | TRUE | Block `COPY INTO 's3://...'` with inline credentials/URLs |
| `PREVENT_LOAD_FROM_INLINE_URL` | TRUE | Block `COPY INTO <table> FROM 's3://...'` with inline URLs |
| `REQUIRE_STORAGE_INTEGRATION_FOR_STAGE_CREATION` | TRUE | Named external stages must use a storage integration (no embedded keys) |
| `REQUIRE_STORAGE_INTEGRATION_FOR_STAGE_OPERATION` | TRUE | Loads and unloads to private cloud storage must go through an integration-backed stage |
| `PREVENT_UNLOAD_TO_INTERNAL_STAGES` | TRUE (if feasible) | Block `COPY INTO @internal_stage` unloads |
| `DISABLE_UI_DOWNLOAD_BUTTON` | TRUE (if policy requires) | Hide result download in Snowsight; can also be set per user |

Restrict which roles can create storage integrations, and use the `STORAGE_ALLOWED_LOCATIONS` / `STORAGE_BLOCKED_LOCATIONS` properties to constrain them.

In [ ]:
%%sql -r EXFILTRATION_PARAMETERS
SHOW PARAMETERS IN ACCOUNT
->> SELECT
        p."key"   AS parameter,
        p."value" AS current_value,
        e.recommended,
        p."level" AS set_at_level,
        CASE WHEN UPPER(p."value") = UPPER(e.recommended) THEN '✅ Matches recommendation'
             ELSE '⚠️ Review' END AS status,
        e.rationale
    FROM $1 p
    JOIN (VALUES
        ('PREVENT_UNLOAD_TO_INLINE_URL', 'true', 'Block ad hoc unload to external URLs'),
        ('PREVENT_LOAD_FROM_INLINE_URL', 'true', 'Block ad hoc load from external URLs'),
        ('REQUIRE_STORAGE_INTEGRATION_FOR_STAGE_CREATION', 'true', 'No credentials embedded in stages'),
        ('REQUIRE_STORAGE_INTEGRATION_FOR_STAGE_OPERATION', 'true', 'Only integration-backed stages for private storage'),
        ('PREVENT_UNLOAD_TO_INTERNAL_STAGES', 'true', 'Block unload to user/table/named internal stages'),
        ('DISABLE_UI_DOWNLOAD_BUTTON', 'true', 'Prevent result downloads from Snowsight')
    ) e(parameter, recommended, rationale) ON p."key" = e.parameter
    ORDER BY status DESC, parameter;

In [ ]:
%%sql -r EXTERNAL_STAGES_WITH_CREDENTIALS
-- External stages: flag those with embedded credentials or no storage integration
SHOW STAGES IN ACCOUNT
->> SELECT
        "database_name", "schema_name", "name", "url", "cloud", "region",
        "storage_integration", "has_credentials", "owner",
        CASE WHEN "has_credentials" = 'Y'           THEN '❌ Embedded credentials'
             WHEN "storage_integration" IS NULL     THEN '⚠️ No storage integration (public bucket?)'
             ELSE '✅ Uses storage integration' END AS status
    FROM $1
    WHERE "type" ILIKE 'EXTERNAL%'
    ORDER BY status, "database_name", "name";

In [ ]:
%%sql -r STORAGE_INTEGRATIONS
SHOW STORAGE INTEGRATIONS;

### 1.7 Authentication and Credential Management

> **Single-factor passwords are being retired.** Per [Planning for the deprecation of single-factor password sign-ins](https://docs.snowflake.com/en/user-guide/security-mfa-rollout):
> - **Phase 1 (Sep 2025 – Jan 2026):** MFA mandatory for all password sign-ins to Snowsight.
> - **Phase 2 (May – Jul 2026):** New human users must use MFA with passwords on every surface; no new `LEGACY_SERVICE` users.
> - **Phase 3 (Aug – Oct 2026):** Human users can no longer authenticate with a password alone; `LEGACY_SERVICE` users are converted to `SERVICE` and lose password access.
>
> Enforcement dates vary by account. The **Strong Authentication Hub** in Snowsight shows your account's dates and affected users.

Recommended authentication by user type:

| User type | Recommended methods |
|---|---|
| Human (`TYPE=PERSON`) | SSO (SAML2 / OIDC) with IdP MFA; otherwise password + Snowflake MFA (passkey preferred) |
| Service (`TYPE=SERVICE`) | Workload Identity Federation (AWS, Azure, GCP, OIDC), OAuth client credentials, key pair, or a PAT restricted by a network policy |

The cells below inventory the account's authentication landscape, combining configuration data with 30–90 days of login history.

In [ ]:
%%sql -r STRONG_AUTH_READINESS
-- Users who will be blocked when single-factor passwords are fully deprecated
SELECT
    name, type, has_password, has_mfa, has_rsa_public_key, has_pat, has_workload_identity,
    ext_authn_uid IS NOT NULL AS has_sso_identity, last_success_login,
    CASE
        WHEN type = 'LEGACY_SERVICE' AND has_password
            THEN '❌ Legacy service user with password - migrate to WIF / key pair / OAuth / PAT'
        WHEN COALESCE(type,'PERSON') = 'PERSON' AND has_password AND NOT has_mfa
            THEN '❌ Human password user without MFA'
        WHEN type = 'LEGACY_SERVICE'
            THEN '⚠️ LEGACY_SERVICE type is being retired - change to SERVICE'
        WHEN type IS NULL
            THEN '⚠️ TYPE not set - set PERSON or SERVICE explicitly'
    END AS finding
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL AND disabled = FALSE
HAVING finding IS NOT NULL
ORDER BY finding, name;

In [ ]:
%%sql -r SINGLE_FACTOR_PASSWORD_LOGINS
-- Successful password logins with no second factor in the last 30 days, by client
SELECT
    l.user_name,
    u.type AS user_type,
    l.reported_client_type,
    COUNT(*)                AS logins_30d,
    MAX(l.event_timestamp)  AS last_login
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY l
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON u.name = l.user_name AND u.deleted_on IS NULL
WHERE l.event_timestamp >= DATEADD(DAY, -30, CURRENT_TIMESTAMP())
  AND l.is_success = 'YES'
  AND l.first_authentication_factor = 'PASSWORD'
  AND l.second_authentication_factor IS NULL
GROUP BY ALL
ORDER BY logins_30d DESC;

In [ ]:
%%sql -r AUTH_METHOD_USAGE
-- Authentication factors in use over the last 90 days
SELECT
    first_authentication_factor,
    COALESCE(second_authentication_factor, '(none)') AS second_authentication_factor,
    COUNT(DISTINCT user_name) AS users,
    COUNT_IF(is_success = 'YES') AS successful_logins,
    COUNT_IF(is_success = 'NO')  AS failed_logins
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE event_timestamp >= DATEADD(DAY, -90, CURRENT_TIMESTAMP())
GROUP BY ALL
ORDER BY users DESC;

In [ ]:
%%sql -r FAILED_LOGIN_HOTSPOTS
-- Users/IPs with repeated failed logins in the last 7 days (possible brute force or broken credentials)
SELECT
    user_name, client_ip, reported_client_type, error_code, error_message,
    COUNT(*) AS failures, MIN(event_timestamp) AS first_seen, MAX(event_timestamp) AS last_seen
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE event_timestamp >= DATEADD(DAY, -7, CURRENT_TIMESTAMP())
  AND is_success = 'NO'
GROUP BY ALL
HAVING COUNT(*) >= 5
ORDER BY failures DESC;

#### Security Integrations (SSO, OAuth, External OAuth)

Review every SAML2, OAuth, and External OAuth integration. Recommended settings:
- **Snowflake OAuth:** `OAUTH_ADD_PRIVILEGED_ROLES_TO_BLOCKED_LIST = TRUE` (the default) so ACCOUNTADMIN, SECURITYADMIN, and ORGADMIN cannot be used through OAuth; keep `OAUTH_ISSUE_REFRESH_TOKENS` and `OAUTH_REFRESH_TOKEN_VALIDITY` as short as practical; enforce PKCE for public clients.
- **External OAuth:** `EXTERNAL_OAUTH_ANY_ROLE_MODE = DISABLE` unless required, and keep `EXTERNAL_OAUTH_ADD_PRIVILEGED_ROLES_TO_BLOCKED_LIST = TRUE`.
- **SAML2:** Disable or remove unused integrations, and track certificate expiry.

In [ ]:
%%sql -r SECURITY_INTEGRATION_INVENTORY
SHOW SECURITY INTEGRATIONS
->> SELECT "name", "type", "category", "enabled", "created_on",
        IFF("enabled" = 'false', '⚠️ Disabled - remove if unused', '✅ Enabled') AS status
    FROM $1
    ORDER BY "type", "name";

In [ ]:
%%sql -r INTEGRATION_USAGE_90D
-- Which security integrations are actually used to log in (90 days)
SELECT
    first_authentication_factor,
    COUNT(DISTINCT user_name) AS users,
    COUNT(*)                  AS successful_logins,
    MAX(event_timestamp)      AS last_used
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE event_timestamp >= DATEADD(DAY, -90, CURRENT_TIMESTAMP())
  AND is_success = 'YES'
  AND (first_authentication_factor ILIKE '%SAML%' OR first_authentication_factor ILIKE '%OAUTH%'
       OR first_authentication_factor ILIKE '%WORKLOAD%' OR first_authentication_factor ILIKE '%OIDC%')
GROUP BY ALL
ORDER BY successful_logins DESC;

In [ ]:
%%sql -r OAUTH_PRIVILEGED_ROLE_PARAMETERS
SHOW PARAMETERS IN ACCOUNT
->> SELECT
        p."key"   AS parameter,
        p."value" AS current_value,
        e.recommended,
        p."level" AS set_at_level,
        CASE WHEN UPPER(p."value") = UPPER(e.recommended) THEN '✅ Matches recommendation'
             ELSE '⚠️ Review' END AS status,
        e.rationale
    FROM $1 p
    JOIN (VALUES
        ('OAUTH_ADD_PRIVILEGED_ROLES_TO_BLOCKED_LIST', 'true', 'Block admin roles via Snowflake OAuth'),
        ('EXTERNAL_OAUTH_ADD_PRIVILEGED_ROLES_TO_BLOCKED_LIST', 'true', 'Block admin roles via External OAuth')
    ) e(parameter, recommended, rationale) ON p."key" = e.parameter
    ORDER BY status DESC, parameter;

#### Authentication Policies

Authentication policies control how users authenticate: allowed `AUTHENTICATION_METHODS`, `CLIENT_TYPES`, `SECURITY_INTEGRATIONS`, MFA enrollment and allowed MFA methods (`MFA_POLICY`), and PAT and WIF settings (`PAT_POLICY`, `WORKLOAD_IDENTITY_POLICY`). Create separate policies for human users, service users, and administrators. A user-level policy overrides the account-level policy.

> **Syntax update:** `MFA_AUTHENTICATION_METHODS` is deprecated in favor of `MFA_POLICY = (ENFORCE_MFA_ON_EXTERNAL_AUTHENTICATION = 'ALL' | 'NONE')`, and `MFA_ENROLLMENT = 'OPTIONAL'` is deprecated (it converts to `REQUIRED_SNOWFLAKE_UI_PASSWORD_ONLY`). Update any automation that creates authentication policies.

In [ ]:
%%sql -r SHOW_AUTHENTICATION_POLICIES
SHOW AUTHENTICATION POLICIES IN ACCOUNT;

In [ ]:
%%sql -r AUTHENTICATION_POLICY_ASSIGNMENTS
SELECT policy_name, ref_entity_domain, ref_entity_name, policy_status
FROM SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES
WHERE policy_kind = 'AUTHENTICATION_POLICY'
ORDER BY ref_entity_domain, ref_entity_name;

In [ ]:
%%sql -r EX_AUTH_POLICY
-- EXAMPLES ONLY - uncomment and adapt. Nothing in this cell runs as-is.

-- Human users: SSO or password, MFA required for password, passkey/TOTP only
-- CREATE AUTHENTICATION POLICY human_users_policy
--     AUTHENTICATION_METHODS = ('SAML', 'PASSWORD')
--     SECURITY_INTEGRATIONS  = ('<SAML_INTEGRATION>')
--     MFA_ENROLLMENT         = 'REQUIRED'
--     MFA_POLICY             = (ALLOWED_METHODS = ('PASSKEY', 'TOTP'));

-- Administrators: also require Snowflake MFA after SSO
-- CREATE AUTHENTICATION POLICY admin_users_policy
--     AUTHENTICATION_METHODS = ('SAML', 'PASSWORD')
--     SECURITY_INTEGRATIONS  = ('<SAML_INTEGRATION>')
--     MFA_ENROLLMENT         = 'REQUIRED'
--     MFA_POLICY             = (ENFORCE_MFA_ON_EXTERNAL_AUTHENTICATION = 'ALL');

-- Service users: workload identity / key pair / OAuth only, from drivers
-- CREATE AUTHENTICATION POLICY service_users_policy
--     AUTHENTICATION_METHODS = ('WORKLOAD_IDENTITY', 'KEYPAIR', 'OAUTH')
--     CLIENT_TYPES           = ('DRIVERS', 'SNOWFLAKE_CLI');

-- ALTER ACCOUNT SET AUTHENTICATION POLICY human_users_policy;
-- ALTER USER <svc_user> SET AUTHENTICATION POLICY service_users_policy;
SELECT 'Example policies are commented out' AS note;

#### Multi-Factor Authentication

Snowflake MFA supports **passkeys** (recommended), **authenticator apps (TOTP)**, and **Duo**. Duo enrollments are not replicated to secondary accounts, so prefer passkeys or TOTP if you rely on replication or failover. Restrict allowed methods with `MFA_POLICY = (ALLOWED_METHODS = (...))`.

For break-glass administrators, generate one-time passcodes (`ALTER USER <user> ADD MFA METHOD OTP COUNT = 5;`) and store them in a vault with the password.

To reduce repeated MFA prompts from drivers, consider `ALLOW_CLIENT_MFA_CACHING = TRUE`. For SSO through the external browser from desktop tools, consider `ALLOW_ID_TOKEN = TRUE`. Both trade some security for usability, so evaluate them against your policy.

In [ ]:
%%sql -r MFA_COVERAGE
-- MFA coverage for human password users
SELECT
    COUNT(*)                         AS human_password_users,
    COUNT_IF(has_mfa)                AS with_mfa,
    COUNT_IF(NOT has_mfa)            AS without_mfa,
    COUNT_IF(ext_authn_duo)          AS duo_enrolled,
    COUNT_IF(bypass_mfa_until > CURRENT_TIMESTAMP()) AS mfa_currently_bypassed,
    ROUND(100 * COUNT_IF(has_mfa) / NULLIF(COUNT(*), 0), 1) AS pct_mfa,
    IFF(COUNT_IF(NOT has_mfa) = 0, '✅ All covered', '❌ Users without MFA') AS status
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL AND disabled = FALSE
  AND COALESCE(type, 'PERSON') = 'PERSON'
  AND has_password;

In [ ]:
%%sql -r MFA_CLIENT_PARAMETERS
SHOW PARAMETERS IN ACCOUNT
->> SELECT
        p."key"   AS parameter,
        p."value" AS current_value,
        e.recommended,
        p."level" AS set_at_level,
        CASE WHEN UPPER(p."value") = UPPER(e.recommended) THEN '✅ Matches recommendation'
             ELSE '⚠️ Review' END AS status,
        e.rationale
    FROM $1 p
    JOIN (VALUES
        ('ALLOW_CLIENT_MFA_CACHING', 'false', 'Caching reduces prompts; enable only if accepted by policy'),
        ('ALLOW_ID_TOKEN', 'false', 'SSO token caching for desktop tools; enable only if accepted by policy'),
        ('DISABLE_MFA_ENROLLMENT_PROMPT', 'false', 'Keep the MFA enrollment nudge on for unenrolled password users')
    ) e(parameter, recommended, rationale) ON p."key" = e.parameter
    ORDER BY status DESC, parameter;

#### Password Policies and Leaked Password Protection

Passwords should only be used for human users with MFA, and as a migration step. Set an account-level password policy that meets your compliance requirements (`PASSWORD_MIN_LENGTH`, `PASSWORD_HISTORY`, `PASSWORD_MIN_AGE_DAYS`, `PASSWORD_MAX_AGE_DAYS`, `PASSWORD_MAX_RETRIES`, `PASSWORD_LOCKOUT_TIME_MINS`).

**Leaked password protection:** Snowflake automatically disables passwords it finds in known leaked-credential sources. Affected users must have their password reset by an administrator.

In [ ]:
%%sql -r SHOW_PW_POLICIES
SHOW PASSWORD POLICIES IN ACCOUNT;

In [ ]:
%%sql -r PASSWORD_AGE
-- Human password users whose password is older than 365 days
SELECT name, type, has_mfa, password_last_set_time, last_success_login
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL AND disabled = FALSE AND has_password
  AND password_last_set_time < DATEADD(DAY, -365, CURRENT_TIMESTAMP())
ORDER BY password_last_set_time;

#### Service User Authentication: WIF, Key Pair, PAT

- **Workload Identity Federation (WIF)** is the preferred method for workloads running on AWS, Azure, or GCP, or on any OIDC issuer (e.g. GitHub Actions or Kubernetes). It uses short-lived, cloud-issued identity, so there are no secrets to store or rotate. Configure it with `ALTER USER ... SET WORKLOAD_IDENTITY = (...)`.
- **Key pair:** Rotate keys using the `RSA_PUBLIC_KEY` / `RSA_PUBLIC_KEY_2` slots, and protect private keys in a secrets manager.
- **Programmatic Access Tokens (PATs):** By default, service users need a network policy before they can use a PAT. Set short expirations, scope each PAT to a role (`ROLE_RESTRICTION`), and constrain PATs with the `PAT_POLICY` in authentication policies.

Apply a network policy to every service user, whatever the method.

In [ ]:
%%sql -r SERVICE_USER_AUTH_METHODS
SELECT
    name, type, has_workload_identity, has_rsa_public_key, has_pat, has_password, last_success_login,
    CASE
        WHEN has_password THEN '❌ Password on service user'
        WHEN NOT (has_workload_identity OR has_rsa_public_key OR has_pat) THEN '⚠️ No credential configured'
        WHEN has_workload_identity THEN '✅ Workload identity'
        ELSE '✅ Key pair / PAT' END AS status
FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
WHERE deleted_on IS NULL AND disabled = FALSE
  AND type IN ('SERVICE', 'LEGACY_SERVICE')
ORDER BY status, name;

In [ ]:
%%sql -r PAT_INVENTORY
-- Programmatic Access Token lifecycle review
SELECT
    c.user_name,
    c.name               AS token_name,
    c.status,
    c.created_on,
    c.expiration_date,
    c.last_used_on,
    DATEDIFF(DAY, c.created_on, c.expiration_date) AS lifetime_days,
    c.additional_details:role_restriction::STRING AS role_restriction,
    CASE
        WHEN c.expiration_date < CURRENT_TIMESTAMP()                                THEN '⚠️ Expired - remove'
        WHEN c.last_used_on IS NULL AND c.created_on < DATEADD(DAY, -30, CURRENT_TIMESTAMP())
                                                                                     THEN '⚠️ Never used - revoke'
        WHEN c.last_used_on < DATEADD(DAY, -90, CURRENT_TIMESTAMP())                THEN '⚠️ Unused 90+ days - revoke'
        WHEN c.expiration_date < DATEADD(DAY, 30, CURRENT_TIMESTAMP())              THEN '👀 Expires within 30 days'
        WHEN DATEDIFF(DAY, c.created_on, c.expiration_date) > 90                    THEN '⚠️ Long-lived token (>90 days)'
        ELSE '✅ OK' END AS review_status
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS c
WHERE c.type = 'PAT'
ORDER BY review_status, c.expiration_date;

### 1.8 Access Control: Managed Access Schemas and Grants

In a **managed access schema**, object owners lose the ability to grant access. Only the schema owner or a role with `MANAGE GRANTS` can grant privileges on objects in the schema, which centralizes privilege management. Use managed access for schemas that contain sensitive data.

In [ ]:
%%sql -r MANAGED_ACCESS_SCHEMA_VERIFY
-- Managed access coverage per database (all databases in the account)
SELECT
    catalog_name AS database_name,
    COUNT(*)                         AS schemas,
    COUNT_IF(is_managed_access = 'YES') AS managed_access_schemas,
    ROUND(100 * COUNT_IF(is_managed_access = 'YES') / COUNT(*), 1) AS pct_managed
FROM SNOWFLAKE.ACCOUNT_USAGE.SCHEMATA
WHERE deleted IS NULL
  AND schema_name NOT IN ('INFORMATION_SCHEMA', 'PUBLIC')
GROUP BY ALL
ORDER BY pct_managed, database_name;

In [ ]:
%%sql -r PUBLIC_ROLE_GRANTS
-- Privileges granted to PUBLIC (available to every user)
SELECT privilege, granted_on, table_catalog, table_schema, name, granted_by, created_on
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE grantee_name = 'PUBLIC' AND deleted_on IS NULL
ORDER BY granted_on, table_catalog, name;

#### Pre-signed URLs

`GET_PRESIGNED_URL` creates a time-limited URL for a file on a stage, and Snowflake cannot audit downloads made through that URL. To disable the function, open a Support ticket to set `ENABLE_GET_PRESIGNED_URL = FALSE`.

### 1.9 External Access Integrations, Secrets, and Egress

External access integrations allow UDFs, procedures, notebooks, and containers to reach external endpoints listed in egress network rules, optionally using secrets. Review:
- Which network rules (hosts) each integration allows. Avoid wildcard hosts such as `0.0.0.0` or `*.com`.
- Which roles have `USAGE` on each integration and secret.

In [ ]:
%%sql -r EXTERNAL_ACCESS_VALIDATION
SHOW EXTERNAL ACCESS INTEGRATIONS;

In [ ]:
%%sql -r EGRESS_NETWORK_RULES
SHOW NETWORK RULES IN ACCOUNT
->> SELECT "database_name", "schema_name", "name", "type", "mode", "entries_in_valuelist", "owner"
    FROM $1 WHERE "mode" = 'EGRESS';

In [ ]:
%%sql -r SECRETS_INVENTORY
SHOW SECRETS IN ACCOUNT;

### 1.10 Cortex AI and AI Agent Governance

By default, the `SNOWFLAKE.CORTEX_USER` database role is granted to `PUBLIC`, so every role can call Cortex AI functions. To restrict access, revoke it and grant it (or the narrower `CORTEX_EMBED_USER` / `CORTEX_AGENT_USER` roles) only to approved roles:

```sql
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE <approved_role>;
```

Account parameters to review:
- `CORTEX_MODELS_ALLOWLIST`: Restrict which LLMs may be used (`ALL`, `NONE`, or a list).
- `CORTEX_ENABLED_CROSS_REGION`: Whether inference may run outside your account's region. This matters for data residency.
- Cortex Code daily credit limits per user (`CORTEX_CODE_*_DAILY_EST_CREDIT_LIMIT_PER_USER`).
- `CORTEX_MCP_SERVER_FORCE_SQL_EXEC_READ_ONLY`: Force MCP server SQL tools to be read-only.

Also review Cortex Agents and MCP servers: who owns them, which tools they expose, and which roles can use them.

In [ ]:
%%sql -r CORTEX_USER_GRANTS
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_USER
->> SELECT "granted_to", "grantee_name", "granted_by", "created_on",
        IFF("grantee_name" = 'PUBLIC', '⚠️ Granted to PUBLIC - all roles can use Cortex', '✅') AS status
    FROM $1;

In [ ]:
%%sql -r CORTEX_PARAMETERS
SHOW PARAMETERS LIKE 'CORTEX%' IN ACCOUNT
->> SELECT "key", "value", "default", "level", "description" FROM $1 ORDER BY "key";

In [ ]:
%%sql -r CORTEX_AGENTS
SHOW AGENTS IN ACCOUNT;

### 1.11 Error Message Redaction

`ENABLE_UNREDACTED_QUERY_SYNTAX_ERROR` and `ENABLE_UNREDACTED_SECURE_OBJECT_ERROR` can expose query text or secure-object internals in error messages and query history. Keep them FALSE unless they are needed for troubleshooting.

In [ ]:
%%sql -r REDACTION_PARAMETERS
SHOW PARAMETERS IN ACCOUNT
->> SELECT
        p."key"   AS parameter,
        p."value" AS current_value,
        e.recommended,
        p."level" AS set_at_level,
        CASE WHEN UPPER(p."value") = UPPER(e.recommended) THEN '✅ Matches recommendation'
             ELSE '⚠️ Review' END AS status,
        e.rationale
    FROM $1 p
    JOIN (VALUES
        ('ENABLE_UNREDACTED_QUERY_SYNTAX_ERROR', 'false', 'Avoid exposing malformed query text'),
        ('ENABLE_UNREDACTED_SECURE_OBJECT_ERROR', 'false', 'Avoid exposing secure object internals')
    ) e(parameter, recommended, rationale) ON p."key" = e.parameter
    ORDER BY status DESC, parameter;

## Section 2: Enterprise Edition Security Features

### 2.1 Time Travel and Retention

Time Travel can be set from 0 to 90 days per object on Enterprise Edition (the default is 1 day), plus 7 days of Fail-safe for permanent tables. Your legal and audit teams may care about how long data can be recovered (e.g. for GDPR). Align `DATA_RETENTION_TIME_IN_DAYS` and `MIN_DATA_RETENTION_TIME_IN_DAYS` with your retention policy. For ransomware resilience, also consider **backups** (backup sets with retention locks).

In [ ]:
%%sql -r TIME_TRAVEL_ACCOUNT
SHOW PARAMETERS LIKE '%DATA_RETENTION_TIME_IN_DAYS' IN ACCOUNT;

In [ ]:
%%sql -r TIME_TRAVEL_DBS
SELECT database_name, database_owner, retention_time, type, comment
FROM SNOWFLAKE.ACCOUNT_USAGE.DATABASES
WHERE deleted IS NULL
ORDER BY retention_time, database_name;

In [ ]:
%%sql -r TIME_TRAVEL_DISTRIBUTION
-- Distribution of table retention settings (avoid listing every table)
SELECT
    table_catalog, retention_time, is_transient, COUNT(*) AS tables
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE deleted IS NULL AND table_schema <> 'INFORMATION_SCHEMA' AND table_type = 'BASE TABLE'
GROUP BY ALL
ORDER BY table_catalog, retention_time;

### 2.2 Periodic Data Rekeying

On Enterprise Edition and higher, Snowflake can re-encrypt data that was encrypted more than a year ago, so no single key protects data for longer than a year.

In [ ]:
%%sql -r PERIODIC_DATA_REKEYING_VERIFY
SHOW PARAMETERS IN ACCOUNT
->> SELECT
        p."key"   AS parameter,
        p."value" AS current_value,
        e.recommended,
        p."level" AS set_at_level,
        CASE WHEN UPPER(p."value") = UPPER(e.recommended) THEN '✅ Matches recommendation'
             ELSE '⚠️ Review' END AS status,
        e.rationale
    FROM $1 p
    JOIN (VALUES
        ('PERIODIC_DATA_REKEYING', 'true', 'Rekey data older than one year')
    ) e(parameter, recommended, rationale) ON p."key" = e.parameter
    ORDER BY status DESC, parameter;

### 2.3 Data Classification, Tagging, and Masking

1. **Classify**: Use [sensitive data classification](https://docs.snowflake.com/en/user-guide/classify-intro) with a classification profile attached to databases or schemas, so new tables are classified automatically and `SNOWFLAKE.CORE.SEMANTIC_CATEGORY` / `PRIVACY_CATEGORY` tags are applied.
2. **Tag**: Map classification results to your own governance tags.
3. **Protect**: Attach masking policies to tags (tag-based masking) so new columns are protected as soon as they are tagged. Use row access, projection, and aggregation policies where appropriate.

In [ ]:
%%sql -r CLASSIFICATION_COVERAGE
-- Columns carrying system classification tags
SELECT tag_name, tag_value, COUNT(*) AS columns, COUNT(DISTINCT object_database) AS databases
FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
WHERE tag_database = 'SNOWFLAKE' AND tag_schema = 'CORE'
  AND tag_name IN ('SEMANTIC_CATEGORY', 'PRIVACY_CATEGORY')
  AND object_deleted IS NULL
GROUP BY ALL
ORDER BY columns DESC;

In [ ]:
%%sql -r CLASSIFICATION_PROFILES
SHOW SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE IN ACCOUNT;

In [ ]:
%%sql -r TAGS_IN_ACCOUNT
SHOW TAGS IN ACCOUNT;

In [ ]:
%%sql -r POLICY_COVERAGE
-- Governance policy attachments by kind (masking, row access, projection, aggregation, join)
SELECT
    policy_kind,
    COUNT(DISTINCT policy_name)  AS policies,
    COUNT(*)                     AS attachments,
    COUNT_IF(tag_name IS NOT NULL) AS via_tag
FROM SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES
WHERE policy_kind IN ('MASKING_POLICY','ROW_ACCESS_POLICY','PROJECTION_POLICY','AGGREGATION_POLICY','JOIN_POLICY')
GROUP BY ALL
ORDER BY attachments DESC;

In [ ]:
%%sql -r UNPROTECTED_SENSITIVE_COLUMNS
-- Columns classified as identifiers / quasi-identifiers that have no masking policy
SELECT t.object_database, t.object_schema, t.object_name, t.column_name, t.tag_value AS privacy_category
FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES t
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.POLICY_REFERENCES p
       ON p.policy_kind = 'MASKING_POLICY'
      AND p.ref_database_name = t.object_database
      AND p.ref_schema_name   = t.object_schema
      AND p.ref_entity_name   = t.object_name
      AND p.ref_column_name   = t.column_name
WHERE t.tag_database = 'SNOWFLAKE' AND t.tag_schema = 'CORE' AND t.tag_name = 'PRIVACY_CATEGORY'
  AND t.tag_value IN ('IDENTIFIER', 'QUASI_IDENTIFIER')
  AND t.domain = 'COLUMN'
  AND t.object_deleted IS NULL
  AND p.policy_name IS NULL
ORDER BY 1, 2, 3, 4;

### 2.4 Auditing: Access History

`ACCESS_HISTORY` (Enterprise Edition) records which objects and columns each query read or wrote, so you can audit sensitive data access. The query below lists the most-read objects that carry a classification tag.

In [ ]:
%%sql -r SENSITIVE_OBJECT_ACCESS
WITH sensitive AS (
    SELECT DISTINCT object_database || '.' || object_schema || '.' || object_name AS fqn
    FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
    WHERE tag_database = 'SNOWFLAKE' AND tag_schema = 'CORE' AND tag_name = 'PRIVACY_CATEGORY'
      AND object_deleted IS NULL
)
SELECT
    obj.value:objectName::STRING AS object_name,
    COUNT(DISTINCT ah.user_name) AS distinct_users,
    COUNT(*)                     AS queries_7d
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY ah,
     LATERAL FLATTEN(ah.base_objects_accessed) obj
JOIN sensitive s ON s.fqn = obj.value:objectName::STRING
WHERE ah.query_start_time >= DATEADD(DAY, -7, CURRENT_TIMESTAMP())
GROUP BY ALL
ORDER BY queries_7d DESC
LIMIT 50;

## Section 3: Business Critical Edition Features

### 3.1 Private Connectivity

Decide whether connections use the public endpoint (the default) or private connectivity (AWS PrivateLink, Azure Private Link, or GCP Private Service Connect). To use private connectivity exclusively, attach network policies built on `PRIVATE_HOST_PORT` / `AWSVPCEID` / `AZURELINKID` network rules that block public access.

Also review private connectivity for **Snowflake apps** (Streamlit, Snowpark Container Services) with `ENABLE_PER_ACCOUNT_APP_SERVICE_PRIVATELINK_URL`, and **outbound private connectivity** for external functions and external access.

In [ ]:
%%sql -r PRIVATELINK_CONFIG
SELECT SYSTEM$GET_PRIVATELINK_CONFIG() AS privatelink_config;

In [ ]:
%%sql -r LIST_AUTHORIZED_PRIVATEENDPOINTS
SELECT SYSTEM$GET_PRIVATELINK_AUTHORIZED_ENDPOINTS() AS authorized_endpoints;

### 3.2 Internal Stage Private Connectivity

By default, internal stages are reached through the cloud provider's public storage endpoint.
- **AWS:** Use S3 interface endpoints. To restrict internal-stage access, enable `ENFORCE_NETWORK_RULES_FOR_INTERNAL_STAGES` and attach network rules with `MODE = INTERNAL_STAGE`, using `AWSVPCEID` or IPv4 types.
- **Azure:** Use private endpoints for internal stages (`SYSTEM$AUTHORIZE_STAGE_PRIVATELINK_ACCESS`), then block public access with `SYSTEM$BLOCK_INTERNAL_STAGES_PUBLIC_ACCESS()`, or `..._WITH_EXCEPTION('<ip/cidr>', ...)` to allow specific IPs.
- **GCP:** Use Private Service Connect endpoints for internal stages (`SYSTEM$AUTHORIZE_STAGE_PRIVATELINK_ACCESS`).

Run only the check that matches your cloud. The Azure status function can take a few minutes to run.

In [ ]:
%%sql -r INTERNAL_STAGE_AWS
-- AWS
SHOW PARAMETERS LIKE 'ENFORCE_NETWORK_RULES_FOR_INTERNAL_STAGES' IN ACCOUNT;

In [ ]:
%%sql -r INTERNAL_STAGE_AZURE
-- AZURE / GCP ONLY - uncomment the line(s) for your cloud (these functions do not exist on AWS)
-- SELECT SYSTEM$INTERNAL_STAGES_PUBLIC_ACCESS_STATUS();        -- Azure only
-- SELECT SYSTEM$GET_STAGE_PRIVATELINK_AUTHORIZED_ENDPOINTS();  -- Azure and GCP
SELECT CURRENT_REGION() AS region, 'Uncomment the Azure/GCP check above if applicable' AS note;

### 3.3 Tri-Secret Secure (Customer-Managed Keys)

Tri-Secret Secure combines a customer-managed key (CMK) in your cloud KMS with a Snowflake-maintained key, so revoking the CMK makes the account's data unreadable. Activation is now **self-service**: register your key with `SYSTEM$REGISTER_CMK_INFO`, grant Snowflake access in your KMS, then call `SYSTEM$ACTIVATE_CMK_INFO`. Use `SYSTEM$GET_CMK_CONFIG` to get the principal and policy to configure in your KMS. You can also connect to the KMS over private connectivity.

Lock down KMS access:
- **AWS:** Restrict the key policy to Snowflake's IAM principal, and add a condition on Snowflake's VPC IDs (from `SYSTEM$GET_SNOWFLAKE_PLATFORM_INFO`).
- **Azure:** Restrict Key Vault access to Snowflake's service principal and subnets.
- **GCP:** Grant only the minimum KMS roles to Snowflake's service account.

In [ ]:
%%sql -r VERIFY_TSS
SELECT SYSTEM$GET_CMK_INFO() AS cmk_info;

In [ ]:
%%sql -r RETURN_SNOWFLAKE_INFO
SELECT SYSTEM$GET_SNOWFLAKE_PLATFORM_INFO() AS platform_info;

### 3.4 Business Continuity and Disaster Recovery

Use **failover groups** to replicate databases *and* account objects (users, roles, grants, warehouses, integrations, and network, password, session, and authentication policies) to a secondary account in another region or cloud. This keeps your security configuration consistent after a failover. **Client Redirect** (connection objects) moves client traffic to the secondary account. Business Critical is required for failover and Client Redirect.

Remember that Duo MFA enrollments are not replicated.

In [ ]:
%%sql -r SHOW_REPLICATION_ACCOUNTS
SHOW REPLICATION ACCOUNTS;

In [ ]:
%%sql -r SHOW_FAILOVER_GROUPS
SHOW FAILOVER GROUPS
->> SELECT "name", "type", "is_primary", "object_types", "allowed_accounts", "replication_schedule", "next_scheduled_refresh",
        IFF(NOT CONTAINS(UPPER("object_types"), 'USERS') OR NOT CONTAINS(UPPER("object_types"), 'ROLES'),
            '⚠️ Users/roles not replicated', '✅ Includes users and roles') AS status
    FROM $1;

In [ ]:
%%sql -r SHOW_CONNECTIONS
SHOW CONNECTIONS IN ACCOUNT;

## Next Steps

- Turn on the Trust Center **Security Essentials** and **CIS Benchmark** scanner packages, with email or webhook notifications.
- Use the **Strong Authentication Hub** to finish migrating off single-factor passwords and `LEGACY_SERVICE` users before Phase 3 reaches your account.
- Re-run this notebook regularly (for example quarterly) and after major configuration changes.
- Send feedback to safe@snowflake.com.